# 5.2 Voice-Controlled Drone Flight (Upload Audio File)

In [ ]:
import os

# Build a new prompt; knowledge base written to aisim_lession26.txt
kg_promot_file = "prompts/aisim_lession52.txt"

kg_prompt = """
Here are some functions you can use to command the drone.

aw.takeoff() - Takes off the drone.
aw.land() - Lands the drone.
aw.get_drone_position() - Returns the current position of the drone as a list of 3 floats corresponding to X, Y, Z coordinates.
aw.fly_to([x, y, z]) - Flies the drone to the specified position, given as a list of three parameters corresponding to X, Y, Z coordinates.
aw.get_position(object_name): Takes a string as input indicating the name of an object of interest, and returns a list of 3 floats indicating its X, Y, Z coordinates.

The following objects exist in the scene, and you should refer to them using these exact names:

Wind Turbine 1, Wind Turbine 2, Solar Panels, Car, Crowd, Tower 1, Tower 2, Tower 3.

Note that when making function calls, use the corresponding English names as follows:
turbine1: Wind Turbine 1
turbine2: Wind Turbine 2
solarpanels: Solar Panels
car: Car
crowd: Crowd
tower1: Tower 1
tower2: Tower 2
tower3: Tower 3
House1: White House
RectLight2_3: Light 3
Stairs: Stairs

For example, to get the position of Wind Turbine 1, write:
aw.get_position("turbine1")
Not:
aw.get_position("Wind Turbine 1")

For axis conventions, we use NED (North-East-Down) coordinate system,
where +X is North, +Y is East, +Z is Down. This means higher altitude corresponds to more negative Z values. If the origin is on the ground, Z is zero at ground level and negative above ground. Flying up means subtracting from the Z axis.

All objects except the drone itself are immovable. Remember there are two turbines and three towers. When there are multiple objects of the same type,
if I don't explicitly specify which one I'm referring to, you should always ask for clarification. Never make assumptions.

Replies should follow this format:
```python
i=1  # output python code here
```
This code assigns a value.


You don't need to worry about importing aw; it's already declared in the environment.
"""
os.makedirs(os.path.dirname(kg_promot_file), exist_ok=True)
pt_file = open(kg_promot_file, "w", encoding="utf-8")
pt_file.write(kg_prompt)
pt_file.close()

In [ ]:
import airsim_agent
my_agent = airsim_agent.AirSimAgent(knowledge_prompt="prompts/aisim_lession52.txt")

In [ ]:
command = "Take off"
python_code = my_agent.process(command, True)  # Execute code
print("python_code: \n", python_code)

In [ ]:
command = "Fly around the Stairs as the center point, orbit the Stairs once, avoid getting within 10 meters of the Stairs due to obstacles, then output the execution status as text and save it to log.txt in the current folder, avoiding encoding issues. Include the runtime duration (2 decimal places) with a prefix label: Task Complete:"
python_code = my_agent.process(command, True)  # Execute code
print("python_code: \n", python_code)

In [ ]:
import sys
sys.path.append('..')
import airsim
import airsim_wrapper
import math
import time

# Record start time
start_time = time.time()
aw = airsim_wrapper.AirSimWrapper()
try:
    # Take off
    aw.takeoff()

    # Get Stairs position
    stairs_pos = aw.get_position("Stairs")

    # Set orbit radius (greater than 10 meters)
    radius = 12
    # Set flight altitude
    flight_height = -5

    # Set number of orbit waypoints
    num_points = 8

    for i in range(num_points):
        # Calculate current point angle
        angle = 2 * math.pi * i / num_points
        # Calculate target point coordinates
        x = stairs_pos[0] + radius * math.cos(angle)
        y = stairs_pos[1] + radius * math.sin(angle)
        z = stairs_pos[2] + flight_height

        # Fly to target point
        aw.fly_to([x, y, z])

    # Land
    aw.land()
    status = "completed successfully"
except Exception as e:
    status = f"failed, reason: {str(e)}"

# Record end time
end_time = time.time()
# Calculate elapsed time, keep 2 decimal places
elapsed_time = round(end_time - start_time, 2)

log_message = f"Process finished status: Orbit Stairs mission {status}, runtime: {elapsed_time} seconds"

# Save status to log.txt in the current folder, specifying UTF-8 encoding
with open('log.txt', 'w', encoding='utf-8') as f:
    f.write(log_message)

## Recognize Speech and Convert to Code Commands for the Language Model



In [ ]:
import recognition_module
# Text command, recorded as audio and uploaded to object storage
file_url = "https://yt-shanghai.tos-cn-shanghai.volces.com/mp3%E8%AF%AD%E9%9F%B3/comand.mp3"
text = recognition_module.process_mp3(file_url)
print(text)



Perform speech recognition on the MP3 file and execute the resulting code

In [ ]:
command = text
python_code = my_agent.process(command, True)  # Execute code
print("python_code: \n", python_code)

Read local text and generate MP3 speech output

In [ ]:
     # Read local text file log.txt and convert to string
import voice_module
text = open("log.txt", "r", encoding="utf-8").read()
print(text)
voice_module.process_text2mp3(text)